# Лекція 11 — pandas глибоке занурення на Stack Overflow Developer Survey 2025

**Курс:** Applied Software Development (Python) 2026 ·

---

## Передумови (Prerequisites)

Ця лекція **повністю самодостатня**: жодної залежності від проєктного коду з лекцій 6–10. Нам не потрібні ані вебфреймворк, ані база даних, ані контейнери — лише Python, Jupyter та один CSV.

Достатньо знань з **Лекцій 1–5**:

- типи даних, змінні, f-strings (Л1)
- колекції: `list`, `dict`, `tuple`, `set` (Л2–Л3)
- генератори списків / словників (Л3)
- функції, `*args`/`**kwargs`, lambda (Л3–Л4)
- базове файлове введення-виведення (Л5)

Плюс встановлений **Jupyter** (notebook, lab або VS Code) і можливість запустити `pip install`.

---

## Датасет, на якому все побудовано

Один CSV — **Stack Overflow Annual Developer Survey 2025** (~49 000 респондентів, 177 країн). Ми завантажимо його з офіційного джерела під ліцензією **ODbL**, почистимо, поріжемо, згрупуємо, з'єднаємо — і побачимо, як виглядає світ розробників у числах.

> **Важливо:** CSV-файл **не** вкомітчений у репозиторій (він завеликий). Ви завантажуєте його самі — див. `README.md` у теці цієї лекції.


## Цілі заняття

Після цієї лекції ви зможете:

1. Пояснити, **коли pandas — правильний інструмент**, а коли варто дивитися в бік DuckDB чи Polars.
2. Розрізнити `Series` та `DataFrame` і будувати їх з чистого Python (без зовнішніх файлів).
3. Завантажити справжній, "брудний" CSV з `pd.read_csv` та почистити його: типи, пропущені значення, незграбні рядкові маркери на кшталт `"Less than 1 year"`.
4. Впевнено користуватися `.loc`, `.iloc`, булевими масками, `.query()`, `.isin()`, `.explode()`, `groupby().agg()`, `merge()`, `pivot_table` та `crosstab`.
5. Застосовувати інтермедіат-патерни: method chaining (`.pipe`, `.assign`), `.apply` / `.map`, та `Categorical` dtype з реальним порівнянням використання пам'яті.

## 1. Для чого pandas — і для чого ні

**pandas** — це Python-бібліотека для роботи з табличними даними. Синтаксично схожа на Excel, але в сто разів швидша і в тисячу разів програмованіша.

### Сильні сторони

- **Векторизовані операції над колонками (vectorization).** `df["a"] + df["b"]` — один виклик у C під капотом, а не Python-цикл. Це зазвичай у 10–1000 разів швидше, ніж `for row in df: ...`.
- **Багатий API для злиття, групування, перетворень.** `groupby`, `merge`, `pivot_table`, `rolling`, `resample` — все вбудовано і добре документовано.
- **Екосистема.** pandas легко інтегрується з matplotlib, seaborn, scikit-learn, Jupyter, Parquet, Excel, різними SQL-базами — одним словом, з усім.

### Слабкі сторони

- **Все в пам'яті, один процес.** Якщо датасет не вміщується в RAM — pandas зламається. Жодних out-of-core чи паралельних обчислень (крім зовнішніх костилів).
- **"Магія" індексів.** `.loc` vs `.iloc`, `SettingWithCopyWarning`, view vs copy — pandas має свою філософію, яку треба опанувати, на початках деякі моменти можуть здаватися специфічними.

**Правило великого пальця:** якщо ваш датасет вміщується в пам'ять (≤ кілька GB) і ви робите з ним переважно **агрегації, перетворення та поєднання** — pandas ідеальний. Якщо ні — дивіться в бік DuckDB чи Polars (Секція 16).


## 2. Series vs DataFrame

Перш ніж завантажити справжні дані, збудуємо обидва основні об'єкти pandas **з чистого Python**, щоб побачити їхню природу.

**Ментальна модель:**

- `Series` — одновимірний масив значень з **індексом** (ярликами).
- `DataFrame` — словник `Series`-ів, які **ділять між собою один Index**.

Тобто `DataFrame` — це таблиця, яку можна подивитися як "набір колонок" (кожна колонка — `Series`).


In [1]:
import pandas as pd

# Series з  Python-списку — pandas сам створить цілочисловий Index 0..N-1
temps = pd.Series([18.5, 21.0, 19.8, 22.3], name="temperature_c")
print(temps)
print("---")
print(f"dtype: {temps.dtype}")
print(f"name:  {temps.name}")
print(f"index: {list(temps.index)}")

0    18.5
1    21.0
2    19.8
3    22.3
Name: temperature_c, dtype: float64
---
dtype: float64
name:  temperature_c
index: [0, 1, 2, 3]


In [3]:
# Series з ярликами (індексами): явно задаємо Index
rating = pd.Series(
    data=[4.7, 4.5, 4.9, 4.2],
    index=["Python", "JavaScript", "Rust", "PHP"],
    name="developer_love",
)
print(rating)
print("---")
# Доступ за ярликом, не за позицією
print("Rust:", rating["Rust"])

Python        4.7
JavaScript    4.5
Rust          4.9
PHP           4.2
Name: developer_love, dtype: float64
---
Rust: 4.9


In [4]:
# DataFrame = dict of Series з одним спільним Index
devs = pd.DataFrame(
    {
        "country": ["Ukraine", "USA", "Poland", "Germany"],
        "years_pro": [5, 9, 4, 12],
        "uses_python": [True, True, False, True],
    }
)
print(devs)
print("---")
# Кожна колонка — Series; індекс 0..3 спільний для всіх колонок
print(type(devs["country"]))
print(devs["country"])

   country  years_pro  uses_python
0  Ukraine          5         True
1      USA          9         True
2   Poland          4        False
3  Germany         12         True
---
<class 'pandas.core.series.Series'>
0    Ukraine
1        USA
2     Poland
3    Germany
Name: country, dtype: object


**Ключова ідея:** коли ми потім напишемо `df["Country"]` над справжнім CSV — повернеться саме `Series`. Коли ми напишемо `df[["Country", "DevType"]]` (зверніть увагу, подвійні дужки) — повернеться `DataFrame` з підмножиною колонок.

## 3. Завантаження даних: pd.read_csv на Stack Overflow Survey 2025

Тепер — до справжнього датасету.

In [ ]:
from pathlib import Path

# ящо виберете інший рік, у нього може бути інша схема колонок
SURVEY_YEAR = 2025
SURVEY_PATH = Path("data") / "survey_results_public.csv"

print(f"Survey рік: {SURVEY_YEAR}")
print(f"Очікуваний шлях: {SURVEY_PATH.resolve()}")
print(f"Файл існує: {SURVEY_PATH.exists()}")

### Якщо CSV відсутній

Якщо `data/survey_results_public.csv` ще не на місці — ось що робити:

1. Відкрийте <https://survey.stackoverflow.co/> і знайдіть посилання **"Download the full data set (CSV)"**.
2. Розпакуйте ZIP-архів.
3. Покладіть `survey_results_public.csv` за шляхом `lectures/11-pandas-analytics/data/survey_results_public.csv`.

Далі в ноутбуці ми **перевіряємо наявність файлу в першій клітинці коду нижче**. Якщо файлу нема — ноутбук не падає, а автоматично переходить у **режим синтетичних даних** (невеликий демо-датасет з такою самою формою колонок), щоб ви могли пройти лекцію end-to-end ще до того, як знайшли хвилину на завантаження справжнього CSV.

> **⚠️ Для реальних висновків — завантажте справжній CSV.** Синтетичні дані потрібні лише для швидкого демо того, як працюють виклики pandas.


In [ ]:
# Колонки, які нам знадобляться протягом лекції.
# Повний CSV має ~90 колонок; завантажимо лише потрібні для швидкості.
USECOLS = [
    "ResponseId",
    "MainBranch",
    "Country",
    "EdLevel",
    "YearsCode",
    "YearsCodePro",
    "DevType",
    "LanguageHaveWorkedWith",
    "LanguageWantToWorkWith",
    "DatabaseHaveWorkedWith",
    "ConvertedCompYearly",
    "RemoteWork",
]


def _build_synthetic(n: int = 200) -> pd.DataFrame:
    """Fallback mini-dataset з такою самою формою колонок.
    Використовується, якщо справжній CSV не знайдено."""
    import random

    random.seed(42)
    countries = ["Ukraine", "United States of America", "Germany", "Poland",
                 "India", "United Kingdom", "Brazil", "Canada"]
    ed_levels = ["Primary/elementary school",
                 "Secondary school",
                 "Bachelor's degree (B.A., B.S., B.Eng., etc.)",
                 "Master's degree (M.A., M.S., M.Eng., MBA, etc.)",
                 "Professional degree (JD, MD, Ph.D, Ed.D, etc.)",
                 "Something else"]
    dev_types = ["Developer, back-end", "Developer, front-end",
                 "Developer, full-stack", "Data scientist or machine learning specialist",
                 "Student", "DevOps specialist"]
    langs_pool = ["Python", "JavaScript", "TypeScript", "Go", "Rust", "Java",
                  "C#", "C++", "Ruby", "PHP", "HTML/CSS", "SQL", "Bash/Shell"]
    dbs_pool = ["PostgreSQL", "MySQL", "SQLite", "MongoDB", "Redis", "Oracle"]
    main_branches = ["I am a developer by profession",
                     "I am learning to code",
                     "I used to be a developer"]
    years_pool = [str(i) for i in range(1, 40)] + ["Less than 1 year",
                                                     "More than 50 years"]
    remote_opts = ["Remote", "Hybrid (some remote, some in-person)", "In-person"]

    def pick_many(pool, k_min=1, k_max=4):
        k = random.randint(k_min, min(k_max, len(pool)))
        return ";".join(random.sample(pool, k))

    rows = []
    for rid in range(1, n + 1):
        comp = None if random.random() < 0.30 else random.randint(8_000, 250_000)
        rows.append({
            "ResponseId": rid,
            "MainBranch": random.choice(main_branches),
            "Country": random.choice(countries),
            "EdLevel": random.choice(ed_levels) if random.random() > 0.05 else None,
            "YearsCode": random.choice(years_pool),
            "YearsCodePro": random.choice(years_pool) if random.random() > 0.20 else None,
            "DevType": random.choice(dev_types),
            "LanguageHaveWorkedWith": pick_many(langs_pool, 1, 5),
            "LanguageWantToWorkWith": pick_many(langs_pool, 1, 5),
            "DatabaseHaveWorkedWith": pick_many(dbs_pool, 1, 3),
            "ConvertedCompYearly": comp,
            "RemoteWork": random.choice(remote_opts),
        })
    return pd.DataFrame(rows)


if SURVEY_PATH.exists():
    df = pd.read_csv(
        SURVEY_PATH,
        usecols=USECOLS,
        na_values=["NA", "N/A", ""],
        low_memory=False,
    )
    DATA_MODE = "real"
    print(f"✅ Завантажено справжній Survey 2025: {len(df):,} рядків × {df.shape[1]} колонок")
else:
    df = _build_synthetic()
    DATA_MODE = "synthetic"
    print("⚠️  Справжній CSV не знайдено — працюємо із синтетичним міні-датасетом "
          f"({len(df)} рядків). Дивіться README, щоб завантажити справжній.")


In [ ]:
# Базова "розвідка" датасету
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T.head(15)

In [ ]:
# Розмір, колонки, типи
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("---")
print(df.dtypes)

## 4. Чистка даних: типи та пропущені значення

![pandas: first time?](assets/memes/first-time-pandas.png)

*(Якщо картинка не відображається — це нормально, мем — педагогічний гарнір. Головне — код нижче.)*

Справжні CSV завжди брудні. Дві найчастіші проблеми:

1. **Неправильні типи.** Колонка, яку ми очікуємо числовою (`YearsCodePro`), приїжджає як `object` (рядок), бо в ній сидять значення на кшталт `"Less than 1 year"` або `"More than 50 years"`.
2. **Пропущені значення (NaN).** Респонденти не зобов'язані відповідати на кожне питання.

> **Примітка про дати:** у 2025 Survey **нема колонки з датою/часом відповіді**, тож `pd.to_datetime` ми тут не показуємо — повернемося до нього в лекції, де є справжні часові ряди.

### 4.1. Скільки NaN у кожній колонці?


In [ ]:
df.isna().sum().sort_values(ascending=False)

### 4.2. "Less than 1 year" — чистимо YearsCode та YearsCodePro

Обидві колонки містять текстові "граничні" маркери, які ми хочемо перетворити на числа, не втрачаючи інформацію.


In [ ]:
# Нормалізуємо текстові маркери → числові значення
replacements = {
    "Less than 1 year": 0.5,
    "More than 50 years": 51.0,
}

df["YearsCode"] = pd.to_numeric(
    df["YearsCode"].replace(replacements), errors="coerce"
)
df["YearsCodePro"] = pd.to_numeric(
    df["YearsCodePro"].replace(replacements), errors="coerce"
)

print(df[["YearsCode", "YearsCodePro"]].dtypes)
df[["YearsCode", "YearsCodePro"]].describe()

### 4.3. Що робити з NaN: dropna vs fillna

Два підходи, і вибір залежить від питання:

- `dropna(subset=[...])` — "мене цікавлять лише респонденти, які відповіли на конкретне питання".
- `fillna(value)` — "відсутня відповідь насправді означає щось відоме, і я хочу це кодувати".


In [ ]:
# Drop будь-які рядки, де нема YearsCodePro — бо для наших агрегацій потрібен стаж
n_before = len(df)
df_with_pro = df.dropna(subset=["YearsCodePro"])
n_after = len(df_with_pro)
print(f"dropna(subset=['YearsCodePro']): {n_before:,} → {n_after:,} рядків "
      f"(викинули {n_before - n_after:,})")

In [ ]:
# Fillna: для зручного звіту замінимо відсутній DevType на явну мітку
dev_type_filled = df["DevType"].fillna("— не вказано —")
dev_type_filled.value_counts().head(10)

## 5. Індексація та вибірка

Чотири інструменти, які закривають 95% випадків:

- **`.loc[rows, cols]`** — за ярликами (label-based).
- **`.iloc[rows, cols]`** — за позиціями (positional).
- **Булеві маски** — `df[df["Country"] == "Ukraine"]`.
- **`.query("Country == 'Ukraine'")`** — читабельна версія булевої маски, добре комбінується.

### 5.1. .loc і .iloc


In [ ]:
# .loc — за ярликами (індекс + імена колонок)
# Перші 3 рядки (індекси 0, 1, 2) і дві конкретні колонки
df.loc[0:2, ["Country", "YearsCodePro"]]

In [ ]:
# .iloc — за позиціями (0-based, як Python-списки)
# Перші 3 рядки і колонки 0, 2, 4
df.iloc[0:3, [0, 2, 4]]

### 5.2. Булеві маски + пастка з дужками

Комбінувати маски потрібно через `&`, `|`, `~` (не `and`/`or`/`not`!) — **і кожну маску загортати в дужки**. Це класична помилка-новачка.


In [ ]:
# ПРАВИЛЬНО: дужки навколо кожної маски
mask = (df["YearsCodePro"] >= 5) & (df["ConvertedCompYearly"].notna())
n = mask.sum()
print(f"Респондентів з ≥5 років стажу і вказаною компенсацією: {n:,}")

In [ ]:
# ТА САМА вибірка через .query() — зазвичай читабельніше
subset = df.query("YearsCodePro >= 5 and ConvertedCompYearly.notna()")
print(f".query() дає той самий результат: {len(subset):,} рядків")
subset.head(3)

In [ ]:
# .isin() — перевірка належності до множини
top5_countries = ["Ukraine", "United States of America", "Germany", "India", "United Kingdom"]
df_top5 = df[df["Country"].isin(top5_countries)]
df_top5["Country"].value_counts()

## 6. Багатозначні стовпці та `.explode()`

Survey любить зберігати кілька значень в одній клітинці через крапку з комою:

```text
LanguageHaveWorkedWith:  "Python;JavaScript;Go"
```

Щоб порахувати, скільки респондентів використовують кожну мову, треба **розгорнути** це в довгу форму — один рядок на одну мову.

![explode-schematic](assets/diagrams/explode-schematic.png)

Два кроки:

1. `str.split(";")` — перетворює рядок на список.
2. `.explode(col)` — розгортає кожен елемент списку в окремий рядок.


In [ ]:
# Будуємо "довгу" таблицю: один рядок = один респондент × одна мова
languages_long = (
    df.assign(LanguageList=df["LanguageHaveWorkedWith"].str.split(";"))
      .explode("LanguageList")
      .rename(columns={"LanguageList": "Language"})
      [["ResponseId", "Country", "Language"]]
      .dropna(subset=["Language"])
)

print(f"df: {len(df):,} рядків")
print(f"languages_long: {len(languages_long):,} рядків (один респондент → кілька)")
languages_long.head(10)

In [ ]:
# Топ-15 мов серед усіх респондентів
top_langs = languages_long["Language"].value_counts().head(15)
top_langs

## 7. Groupby та агрегації — де ми в цифрах?

`groupby` — найчастіше вживаний інструмент pandas. Патерн завжди один:

```text
df.groupby(<ключ>)[<колонки>].<агрегація>()
```

**У цьому розділі ми використовуємо Україну як наскрізний якір** — побачимо, як наша країна виглядає на глобальному тлі.

### 7.1. Медіанна компенсація — Україна vs Global


In [ ]:
# Цікавий патерн: згрупувати за булевим, що перетворюється на легку мітку
ua_mask = df["Country"] == "Ukraine"
ua_vs_global = (
    df.groupby(ua_mask.map({True: "Україна", False: "Глобально"}))
      ["YearsCodePro"]
      .median()
)
ua_vs_global

### 7.2. Один ключ: медіанна компенсація за країною (топ-20)

In [ ]:
median_comp_by_country = (
    df.dropna(subset=["ConvertedCompYearly"])
      .groupby("Country")["ConvertedCompYearly"]
      .median()
      .sort_values(ascending=False)
      .head(20)
)
median_comp_by_country

### 7.3. Два ключі + named aggregation

`agg(new_col=("col", "func"))` — синтаксис, який робить вивід охайним і самодокументованим.


In [ ]:
# Медіанна компенсація та кількість респондентів — по країні + рівню освіти
# Обмежимося країнами з достатньою вибіркою, щоб числа щось значили
big_countries = df["Country"].value_counts().head(10).index.tolist()

summary = (
    df[df["Country"].isin(big_countries)]
      .dropna(subset=["ConvertedCompYearly", "EdLevel"])
      .groupby(["Country", "EdLevel"])
      .agg(
          n=("ResponseId", "size"),
          median_usd=("ConvertedCompYearly", "median"),
      )
      .reset_index()
)
summary.head(15)

### 7.4. dropna=False — коли NaN як окрема група

За замовчуванням `groupby` **викидає NaN ключі**. Якщо для вас "не вказано" — самостійна група, передайте `dropna=False`.


In [ ]:
# Скільки респондентів у кожній категорії DevType, включно з NaN
df.groupby("DevType", dropna=False)["ResponseId"].count().head(10)

## 8. Merge / join — зливаємо дві таблиці

Часто треба приєднати результат групування назад до рядків-респондентів ("для кожного респондента покажи, яка середня компенсація у його країні"). Саме для цього `merge`.

Ключові параметри:

- `how=` — `"inner"` (за замовчуванням), `"left"`, `"right"`, `"outer"`.
- `on=` / `left_on=` / `right_on=` — за якою колонкою з'єднуємо.
- `validate=` — перевірка форми з'єднання. Врятує вас від випадкового `many-to-many`, який несподівано перетворює 50 000 рядків на 500 000.

![merge explosion validate](assets/memes/merge-explosion-validate.png)

### 8.1. Inner join: мови → статистика → назад до респондентів


In [ ]:
# Крок 1 — побудувати per-language агрегати
lang_stats = (
    languages_long.groupby("Language")
                  .agg(n_users=("ResponseId", "nunique"))
                  .reset_index()
                  .sort_values("n_users", ascending=False)
)
lang_stats.head(10)

In [ ]:
# Крок 2 — приєднати ці агрегати до кожного рядка languages_long
# Inner merge: тільки збіги (що у нас і є, бо lang_stats побудовано з languages_long)
enriched = languages_long.merge(
    lang_stats,
    on="Language",
    how="inner",
    validate="many_to_one",  # ← КЛЮЧОВА перевірка: одна мова → одна статистика
)
print(f"До merge: {len(languages_long):,} рядків")
print(f"Після merge: {len(enriched):,} рядків (те саме число — bo many_to_one)")
enriched.head(5)

### 8.2. Left join: зберегти всіх, навіть тих, кому пари нема

In [ ]:
# Невелика допоміжна таблиця: "рекомендації" для мов (синтетичне, для демо)
recs = pd.DataFrame({
    "Language": ["Python", "JavaScript", "Rust"],
    "Recommendation": ["✓ solid", "✓ essential", "★ growing"],
})

with_recs = lang_stats.head(10).merge(
    recs, on="Language", how="left", validate="one_to_one",
)
with_recs

**Дивіться на NaN у колонці `Recommendation`:** це саме те, що робить `how="left"` — зберігає всі рядки з лівої таблиці, а праві колонки заповнює `NaN`, де пари нема. Якби ми поставили `how="inner"`, ці рядки просто зникли б.

### 8.3. Коли `validate=` рятує життя

Якщо ви забудете `validate="many_to_one"` і випадково зіллєте два `languages_long`-подібні фрейми (де одна мова має багато записів з **обох** сторін) — ви отримаєте many-to-many і рядки буквально перемножаться. `validate=` кидає виняток одразу, замість того щоб тиша + 10 мільйонів рядків.


## 9. Pivot tables та crosstab

`pivot_table` — "що показати в клітинках таблиці, де рядки — одне, а колонки — інше?".

`crosstab` — частковий випадок `pivot_table` з лічильниками; зручно з `normalize=` для часток.


In [ ]:
# Pivot: медіанна компенсація, рядки = Country, колонки = RemoteWork
pivot = df.pivot_table(
    index="Country",
    columns="RemoteWork",
    values="ConvertedCompYearly",
    aggfunc="median",
    margins=True,          # додає рядок/колонку "All"
    margins_name="Усі",
)
# Обмежимо вивід кількома країнами, щоб було компактно
shown = ["Ukraine", "United States of America", "Germany", "India", "Усі"]
pivot.loc[[c for c in shown if c in pivot.index]]

In [ ]:
# Crosstab з normalize="index": "яка частка мов 'хочу вивчити' у кожному DevType?"
# Спершу збудуємо довгу таблицю для "хочу вивчити"
want_long = (
    df.assign(WantList=df["LanguageWantToWorkWith"].str.split(";"))
      .explode("WantList")
      .rename(columns={"WantList": "WantLanguage"})
      .dropna(subset=["WantLanguage", "DevType"])
)

# Фокус: частка тих, хто хоче Rust, за DevType
want_long["wants_rust"] = want_long["WantLanguage"] == "Rust"
rust_share = pd.crosstab(
    want_long["DevType"],
    want_long["wants_rust"],
    normalize="index",
).rename(columns={True: "Хоче Rust", False: "Не хоче Rust"})
rust_share.sort_values("Хоче Rust", ascending=False).head(10)

## 10. Сортування, ранжування, top-N

Три патерни:

- `.sort_values(by=[...], ascending=[...])` — класичне сортування.
- `.nlargest(N, by=...)` / `.nsmallest(N, by=...)` — швидший і чистіший спосіб взяти top-N, ніж `sort + head`.
- `.rank()` — коли треба впорядкувати та зберегти номер позиції (а не просто відсіяти).


In [ ]:
# Топ-10 країн за кількістю респондентів
country_counts = df["Country"].value_counts()
country_counts.nlargest(10)

In [ ]:
# Те саме, але через sort_values — еквівалентно, але nlargest зазвичай швидший
country_counts.sort_values(ascending=False).head(10)

In [ ]:
# Рангування: хто в топ-5, топ-10, топ-20 країн за кількістю респондентів?
ranked = country_counts.rank(method="min", ascending=False).astype(int)
ranked.head(5)

## 11. Method chaining: `.pipe` та `.assign`

Коли pipeline виростає — стиль із проміжними змінними (`df1 = ...; df2 = ...; df3 = ...`) створює смислові "острівці", які легко сплутати. **Method chaining** (ланцюжок викликів) робить pipeline одним логічним блоком.

### 11.1. Stepwise vs chained — той самий pipeline

**Завдання:** з основного DataFrame взяти респондентів-розробників за професією з відомою компенсацією, додати бакет `comp_band` кожної тисячі доларів, вибрати кілька колонок.


In [ ]:
# Варіант А — stepwise, з проміжними змінними
step1 = df[df["MainBranch"] == "I am a developer by profession"]
step2 = step1.dropna(subset=["ConvertedCompYearly"])
step3 = step2.assign(comp_band=(step2["ConvertedCompYearly"] // 25_000) * 25_000)
step4 = step3[["Country", "DevType", "YearsCodePro", "comp_band"]]

print(step4.shape)
step4.head(5)

In [ ]:
# Варіант Б — chained, один блок. Ідентичний результат.
chained = (
    df.loc[df["MainBranch"] == "I am a developer by profession"]
      .dropna(subset=["ConvertedCompYearly"])
      .assign(comp_band=lambda d: (d["ConvertedCompYearly"] // 25_000) * 25_000)
      [["Country", "DevType", "YearsCodePro", "comp_band"]]
)

print(chained.shape)
chained.head(5)

**Спостереження.** Обидва варіанти дають той самий результат. Chained-варіант:

- не "протікає" проміжними змінними у namespace;
- читається зверху вниз як послідовність кроків;
- дешевше рефакторити (вставити / прибрати крок = змінити один рядок).

**Коли chaining шкодить:** коли ланцюжок стає > 10 кроків, або коли треба дебажити конкретний проміжний результат. Тоді розбивайте назад на stepwise. Або використайте `.pipe(print)` / `.pipe(lambda d: (display(d.head()), d)[1])` як "дебаг-пробник" усередині ланцюжка.

### 11.2. `.pipe(func)` — коли маєте власну функцію-крок


In [ ]:
def drop_outlier_comp(d: pd.DataFrame, lo: float = 1_000, hi: float = 500_000) -> pd.DataFrame:
    """Викинути екстремальні викиди компенсації."""
    return d[(d["ConvertedCompYearly"] >= lo) & (d["ConvertedCompYearly"] <= hi)]


cleaned = (
    df.dropna(subset=["ConvertedCompYearly"])
      .pipe(drop_outlier_comp)
      .assign(comp_band=lambda d: (d["ConvertedCompYearly"] // 25_000) * 25_000)
)
print(f"До: {df.dropna(subset=['ConvertedCompYearly']).shape[0]:,}")
print(f"Після drop_outlier_comp: {cleaned.shape[0]:,}")

## 12. `.apply` та `.map` — власні функції

Коли pandas не має "готової" операції — пишемо свою.

- **`Series.map(dict_or_func)`** — поелементно; найшвидший варіант з тих трьох.
- **`Series.apply(func)`** — поелементно, але приймає функцію; трохи повільніший.
- **`DataFrame.apply(func, axis=0|1)`** — `axis=0` працює по колонках, `axis=1` — по рядках (**найповільніший**).

> **Золоте правило:** перед тим, як тягнутися до `.apply`, запитайте — "чи можна це зробити векторизовано?". Якщо так — робіть векторизовано, буде в 100–1000 разів швидше.


In [ ]:
# .map з dict: переклад довгих офіційних міток на коротші
remote_short = df["RemoteWork"].map({
    "Remote": "remote",
    "Hybrid (some remote, some in-person)": "hybrid",
    "In-person": "onsite",
})
remote_short.value_counts(dropna=False)

In [ ]:
# .apply з lambda: бакет стажу в людські категорії
def years_bucket(y):
    if pd.isna(y):
        return "невідомо"
    if y < 2:
        return "junior (<2)"
    if y < 5:
        return "mid (2–5)"
    if y < 10:
        return "senior (5–10)"
    return "staff+ (10+)"


df["years_bucket"] = df["YearsCodePro"].apply(years_bucket)
df["years_bucket"].value_counts()

### 12.1. Бенчмарк: векторизовано vs `.apply(axis=1)`

Порахуємо "бакет компенсації $25 000" двома способами і заміряємо час.


In [ ]:
# Готуємо підмножину, де є компенсація — щоб було що рахувати
comp_df = df.dropna(subset=["ConvertedCompYearly"]).copy()
print(f"Рядків у бенчмарку: {len(comp_df):,}")

In [ ]:
# Варіант А — векторизовано (одним виразом над цілою колонкою)
%timeit (comp_df["ConvertedCompYearly"] // 25_000) * 25_000

In [ ]:
# Варіант Б — через apply(axis=1): проходимося по кожному рядку в Python
%timeit comp_df.apply(lambda row: (row["ConvertedCompYearly"] // 25_000) * 25_000, axis=1)

**Висновок:** векторизований варіант у десятки чи сотні разів швидший. Саме тому `.apply(axis=1)` — останній притулок, а не перший вибір.


## 13. `Categorical` dtype — пам'ять та впорядковані категорії

`object` (рядковий) тип у колонках на кшталт `Country` або `EdLevel` марнує купу пам'яті: кожен "Ukraine" зберігається як окремий рядок, хоча значень лише кілька десятків.

**Categorical** зберігає унікальні значення одноразово плюс масив цілочислових індексів — як внутрішня "довідникова" таблиця. Переваги:

1. Часто кратне зменшення пам'яті.
2. `groupby` по categorical — швидший.
3. **Впорядковані категорії** (`ordered=True`) дозволяють порівняння на кшталт `edlevel >= "Master's"`.

### 13.1. Пам'ять: before/after на реальних колонках


In [ ]:
# Вимірюємо використання пам'яті об'єктних колонок
def memory_mb(frame: pd.DataFrame) -> float:
    return frame.memory_usage(deep=True).sum() / 1024 ** 2


before_mb = memory_mb(df)
print(f"ДО оптимізації: {before_mb:.2f} MB")

df_opt = df.copy()
for col in ["Country", "DevType", "MainBranch", "RemoteWork"]:
    df_opt[col] = df_opt[col].astype("category")

after_mb = memory_mb(df_opt)
print(f"ПІСЛЯ конвертації 4 колонок у Categorical: {after_mb:.2f} MB")
print(f"Виграш: {(1 - after_mb / before_mb) * 100:.1f}%")

### 13.2. Ordered Categorical: EdLevel як порядкова шкала

`EdLevel` — природна порядкова величина: primary → secondary → bachelor → master → doctorate. Надаючи явний порядок, ми вмикаємо природне порівняння.


In [ ]:
ed_order = [
    "Primary/elementary school",
    "Secondary school",
    "Some college/university study without earning a degree",
    "Associate degree (A.A., A.S., etc.)",
    "Bachelor's degree (B.A., B.S., B.Eng., etc.)",
    "Master's degree (M.A., M.S., M.Eng., MBA, etc.)",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)",
    "Something else",
]
# Беремо лише ті значення, які реально є в датасеті
present = [v for v in ed_order if v in df["EdLevel"].dropna().unique()]
ed_dtype = pd.CategoricalDtype(categories=present, ordered=True)

df_opt["EdLevel"] = df["EdLevel"].astype(ed_dtype)
print("Категорії у порядку:")
for i, c in enumerate(df_opt["EdLevel"].cat.categories, 1):
    print(f"  {i}. {c}")
print("---")
print("Тепер це дає впорядковане порівняння:")
has_master_or_higher = df_opt["EdLevel"] >= "Master's degree (M.A., M.S., M.Eng., MBA, etc.)"
print(f"Респондентів з Master's+ ступенем: {has_master_or_higher.sum():,}")

### 13.3. Україна-якір: фільтр на Categorical

Коли колонка стала Categorical, `df[df["Country"] == "Ukraine"]` працює так само, як і раніше — але під капотом pandas порівнює цілочислові індекси, а не рядки. Швидше та економніше.


In [ ]:
# Фільтр "лише Україна" на Categorical-колонці
ua_respondents = df_opt[df_opt["Country"] == "Ukraine"]
print(f"Респондентів з України: {len(ua_respondents):,}")

# Скільки з них має Master's+?
ua_master_plus = ua_respondents[
    ua_respondents["EdLevel"] >= "Master's degree (M.A., M.S., M.Eng., MBA, etc.)"
]
print(f"З них з Master's+ ступенем: {len(ua_master_plus):,}")

## 14. Практичний блок: три запитання — три пайплайни

Сюди стікається все, що ми вивчили. Кожен пайплайн — ≤ 10 рядків, читабельний зверху вниз.

### 14.1. Топ-10 найпопулярніших мов програмування (глобально)


In [ ]:
top10_langs = (
    df.assign(lang_list=df["LanguageHaveWorkedWith"].str.split(";"))
      .explode("lang_list")
      .rename(columns={"lang_list": "Language"})
      .dropna(subset=["Language"])
      ["Language"]
      .value_counts()
      .head(10)
)
top10_langs

### 14.2. Медіанна компенсація по країнах (топ-20 за кількістю респондентів)

In [ ]:
top20_by_resp = df["Country"].value_counts().head(20).index

med_comp = (
    df[df["Country"].isin(top20_by_resp)]
      .dropna(subset=["ConvertedCompYearly"])
      .groupby("Country")
      .agg(
          n=("ResponseId", "size"),
          median_usd=("ConvertedCompYearly", "median"),
      )
      .sort_values("median_usd", ascending=False)
)
med_comp

### 14.3. Хто хоче вивчити Rust — за DevType

In [ ]:
rust_by_dev = (
    df.assign(wants=df["LanguageWantToWorkWith"].str.split(";"))
      .explode("wants")
      .dropna(subset=["wants", "DevType"])
      .assign(wants_rust=lambda d: d["wants"] == "Rust")
      .groupby("DevType")
      .agg(
          n_respondents=("ResponseId", "nunique"),
          rust_share=("wants_rust", "mean"),
      )
      .sort_values("rust_share", ascending=False)
      .head(10)
)
rust_by_dev

## 15. Коли pandas не підходить

pandas блискуче працює, доки **весь датасет вміщується в RAM** і **вам достатньо одного процесу**. Коли ці умови ламаються — час шукати інше.

| Інструмент | Масштаб | Стиль API | Коли варто |
|---|---|---|---|
| **pandas** | GB в RAM | Імперативний, рядок за рядком | Аналітика, dev-data, прототип моделей |
| **DuckDB** | TB на диску | SQL (+ Python-bind) | OLAP-запити, файли Parquet/CSV без pre-import у БД |
| **Polars** | GB–TB | Lazy, columnar, Rust-backed | Багатопотокові трансформації, великі pipeline-и |

**Коротко:**

- **DuckDB** — "SQL-рушій, який живе у вашому Python-процесі". Ідеальний, коли маєте купу Parquet/CSV на диску і хочете робити SQL-агрегації без підняття окремої БД.
- **Polars** — "pandas, але багатопотокова, ледача та rust-backed". API схожий на pandas, але швидший на великих обсягах і краще масштабується.

У цій лекції ми **не** встановлюємо жодний з них — це концептуальна карта "куди дивитися далі", а не туторіал. Якщо колись ваш pandas-pipeline почне гальмувати або лізти за межі пам'яті — тепер ви знаєте, куди зазирнути.


## 16. Міні-проєкт: "Developer Survey Insights"

Три частини, які ви робите на цій же таблиці `df`. **Це і є міні-проєкт цієї лекції** — замість розрізнених вправ ви будуєте одну зв'язну аналітичну історію.

- **Частина 1 — в аудиторії (~10–15 хв):** простий фільтр + агрегація.
- **Частина 2 — в аудиторії (~10–15 хв):** `.explode()` + `value_counts`.
- **Частина 3 — домашнє завдання (~30–60 хв):** відкрите дослідження з обов'язковим `Categorical` + коментар українською.

Тобто ~25 хв в аудиторії + 30–60 хв удома. Відповідь на Частину 3 — еталонне рішення згорнуте в кінці цього ноутбука.


### Частина 1 — Медіанний `YearsCodePro`: Україна vs Глобально

Порахуйте медіанне значення `YearsCodePro` для респондентів з України та порівняйте з глобальною медіаною. Виведіть обидва числа та різницю (у роках).

**Підказка:** Секція 7.1 — `groupby` з булевим ключем.

*Спершу спробуйте самі, потім розгорніть розв'язок нижче.*


In [ ]:
# ==== РОЗВ'ЯЗОК ЧАСТИНИ 1 ====
ua_median_years_pro = df.loc[df["Country"] == "Ukraine", "YearsCodePro"].median()
global_median_years_pro = df["YearsCodePro"].median()
delta_years = ua_median_years_pro - global_median_years_pro

print("Медіана YearsCodePro:")
print(f"  Україна:    {ua_median_years_pro:.1f} років")
print(f"  Глобальна:  {global_median_years_pro:.1f} років")
print(f"  Різниця:   {delta_years:+.1f} років "
      f"({'Україна нижче' if delta_years < 0 else 'Україна вище'})")

### Частина 2 — Топ-10 мов серед тих, хто "вчиться кодити"

Знайдіть 10 найпопулярніших мов програмування серед респондентів з `MainBranch == "I am learning to code"`. Використайте `str.split(";")` + `.explode()`.

**Підказка:** Секція 6 (`.explode`) + Секція 11 (`.nlargest` / `.value_counts`).

*Спершу спробуйте самі, потім розгорніть розв'язок нижче.*


In [ ]:
# ==== РОЗВ'ЯЗОК ЧАСТИНИ 2 ====
learners_top_langs = (
    df[df["MainBranch"] == "I am learning to code"]
      .assign(lang_list=lambda d: d["LanguageHaveWorkedWith"].str.split(";"))
      .explode("lang_list")
      .rename(columns={"lang_list": "Language"})
      .dropna(subset=["Language"])
      ["Language"]
      .value_counts()
      .head(10)
)
learners_top_langs

### Частина 3 (домашнє завдання) — Країна × DevType × Компенсація

Оберіть **одну країну**. У межах цієї країни знайдіть **3 найпопулярніші `DevType`**. Для кожного з них обчисліть медіанну річну компенсацію (`ConvertedCompYearly`).

**Обов'язкові вимоги:**

1. Примусово приведіть колонки `Country` та `DevType` до `Categorical` dtype.
2. Виведіть результат як **охайний (tidy) DataFrame** з колонками `country, dev_type, n_respondents, median_usd` (саме такі назви).
3. Додайте коментар українською (3–5 речень) про те, що ви побачили. Коментар має посилатися хоча б на одне число з вашого результату.

**Критерії оцінювання (6 балів максимум):**

| Критерій | Бали |
|---|---|
| Правильність 3-рядкового DataFrame (колонки, типи, значення) | 3 |
| Чистий pandas-pipeline (method chaining або ясний stepwise; без зайвих циклів) | 2 |
| Якість коментаря (3–5 речень українською, посилається на конкретне число) | 1 |

Прохідний бал: **≥ 4 / 6**.

**Еталонне рішення** — згорнуте в самому низу цього ноутбука (Секція 21). Відкривайте лише після спроби!


## 17. Підсумок

За цю лекцію ми пройшли повний шлях від "чистий Python" до "справжня аналітика на живих даних":

- **Основи:** Series, DataFrame, `pd.read_csv` з `usecols=` та `na_values=`.
- **Чистка:** dtype coercion, `isna`/`fillna`/`dropna`, нормалізація "Less than 1 year".
- **Вибірка:** `.loc`, `.iloc`, булеві маски з дужками, `.query()`, `.isin()`.
- **Багатозначні колонки:** `str.split(";")` + `.explode()` — один рядок на одне значення.
- **Агрегації:** `groupby().agg()` з named aggregations, `dropna=False`, multi-key.
- **Об'єднання:** `merge(..., validate=...)` як захист від випадкового many-to-many.
- **Переформатування:** `pivot_table` з `margins`, `crosstab` з `normalize`.
- **Ранжування:** `.nlargest`, `.sort_values`, `.rank`.
- **Інтермедіат-патерни:** method chaining (`.pipe`, `.assign`), `.apply`/`.map` з бенчмарком проти векторизації, `Categorical` dtype з реальним виграшем пам'яті.
- **Кордони:** коли pandas ламається і куди дивитися — DuckDB, Polars.

**Наскрізний якір:** Україна — з'явилася в розділах groupby, Categorical та в Частині 1 міні-проєкту. Ви бачили, як наша країна виглядає на глобальній мапі `YearsCodePro` та освітнього рівня.


## 18. Джерела

### Дані

- **Stack Overflow Annual Developer Survey 2025** — офіційна сторінка: <https://survey.stackoverflow.co/2025/>
  Ліцензія: **ODbL** (Open Database License). Атрибуцію збережено в цьому ноутбуці.
- Методологія 2025 Survey: <https://survey.stackoverflow.co/2025/methodology/>

### pandas

- Офіційна документація: <https://pandas.pydata.org/docs/>
- "10 minutes to pandas": <https://pandas.pydata.org/docs/user_guide/10min.html>
- pandas Cookbook: <https://pandas.pydata.org/docs/user_guide/cookbook.html>
- Wes McKinney, *Python for Data Analysis* (3rd edition, O'Reilly, 2022) — книжка автора pandas, чудове друге джерело.

### Ті, що ми лише згадали (Секція 15)

- **DuckDB:** <https://duckdb.org/docs/>
- **Polars:** <https://pola.rs/>

### Додаткове читання українською / англійською

- Real Python, "pandas GroupBy: Your Guide to Grouping Data in Python": <https://realpython.com/pandas-groupby/>
- KDnuggets, "Pandas: Advanced GroupBy Techniques": <https://www.kdnuggets.com/pandas-advanced-groupby-techniques-for-complex-aggregations>

> **Правило курсу:** у цій лекції, як і у всьому курсі, **не** використовуємо джерел російського походження. Всі посилання вище — офіційні або англомовні спільноти.


## 19. Що далі?

**Лекція 12 — NumPy + векторизація + проста ML "з нуля".**

Наступного разу ми:

- зазирнемо під капот pandas — там NumPy-масиви;
- зрозуміємо, чому векторизація швидша за цикли (ви це вже бачили в бенчмарку Секції 12, тепер побачимо **чому**);
- побудуємо простий класифікатор (logistic regression) власними руками, без scikit-learn;
- познайомимося з metrics (accuracy, precision, recall).

**Лекція 13 — Візуалізація.**

Очищений `df` з Survey 2025, який ми тут зробили, **ще повернеться**: у Л13 ми намалюємо з нього графіки matplotlib / seaborn / plotly. Тож саме тепер добре зберегти ваш pipeline чистим — за два тижні ви його переіспользуєте.

Щасливих групувань! 🐼


---

## 20. Еталонне рішення міні-проєкту (Частина 3)

> ⚠️ **Відкривайте лише після самостійної спроби.** Код нижче — один з прийнятних варіантів. Ваш pipeline може виглядати інакше — головне, щоб виходив такий самий tidy DataFrame з 3 рядками.


In [ ]:
# ==== ЕТАЛОННЕ РІШЕННЯ ЧАСТИНИ 3 ====
# Обираємо країну — тут Україна для ілюстрації, ви можете взяти будь-яку
chosen_country = "Ukraine"

# Переводимо Country і DevType у Categorical (вимога задачі)
df3 = df.copy()
df3["Country"] = df3["Country"].astype("category")
df3["DevType"] = df3["DevType"].astype("category")

# Знаходимо 3 найпопулярніші DevType у цій країні
top3_devtypes = (
    df3.loc[df3["Country"] == chosen_country, "DevType"]
       .value_counts()
       .head(3)
       .index
       .tolist()
)
print(f"Top-3 DevType у {chosen_country}:", top3_devtypes)

# Будуємо охайний (tidy) DataFrame з очікуваними колонками
result_df = (
    df3[(df3["Country"] == chosen_country) & (df3["DevType"].isin(top3_devtypes))]
       .dropna(subset=["ConvertedCompYearly"])
       .groupby("DevType", observed=True)
       .agg(
           n_respondents=("ResponseId", "size"),
           median_usd=("ConvertedCompYearly", "median"),
       )
       .reset_index()
       .assign(country=chosen_country)
       .rename(columns={"DevType": "dev_type"})
       [["country", "dev_type", "n_respondents", "median_usd"]]
       .sort_values("n_respondents", ascending=False)
       .reset_index(drop=True)
)

# Приводимо country до Categorical для повної відповідності вимогам
result_df["country"] = result_df["country"].astype("category")
result_df["dev_type"] = result_df["dev_type"].astype("category")

print(result_df.dtypes)
print("---")
result_df

**Приклад коментаря (3–5 речень українською):**

> У вибірці з України найпоширеніший DevType — повноcteкові розробники, і саме вони мають найвищу медіанну компенсацію серед трьох провідних категорій. Back-end розробники на другому місці за кількістю респондентів, але медіана їхньої компенсації помітно нижча за full-stack. Цікавим є розрив між full-stack і front-end — хоча обсяги респондентів порівнянні, різниця в компенсаціях на рівні %-десятків вказує на те, що full-stack стек наразі краще оплачується на українському ринку. Варто пам'ятати, що вибірка українських респондентів у Survey невелика, тож на ці числа варто дивитися як на орієнтир, а не як на офіційну статистику.

---

🐼 *Кінець лекції 11.* Побачимося на Л12 — NumPy + vectorization + ML з нуля.
